# Debug advert_stat DataFrame

??????? ??? ?????? ??????????? ????????? ?????????? ??? ????????? production-????.

In [8]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('D:/Pytnon_scripts/ba_name')

In [9]:
from src.analytics.read_service import GoogleSheetsReadService

reader = GoogleSheetsReadService()
advert_stat_df = reader.get_advert_stats()

print(f'advert_stat_df loaded: rows={len(advert_stat_df)}, cols={len(advert_stat_df.columns)}')

✅ Успешное подключение к Наш Файл УУ ( Акселерация) -> БД_Рекламная_статистика


2026-05-24 18:15:39.563 | INFO     | src.analytics.read_service:read_tab_as_dataframe:70 - Sheet loaded rows=1807 cols=17 header_row=1


advert_stat_df loaded: rows=1807, cols=17


In [10]:
advert_stat_df.shape

(1807, 17)

In [11]:
advert_stat_df.columns.tolist()

['account',
 'advertId',
 'article_id',
 'date',
 'atbs',
 'canceled',
 'clicks',
 'cpc',
 'cr',
 'ctr',
 'orders',
 'sum',
 'sum_price',
 'views',
 'avg_position',
 'cpm',
 'updated_at']

In [12]:
advert_stat_df.dtypes

account         str
advertId        str
article_id      str
date            str
atbs            str
canceled        str
clicks          str
cpc             str
cr              str
ctr             str
orders          str
sum             str
sum_price       str
views           str
avg_position    str
cpm             str
updated_at      str
dtype: object

In [32]:
advert_stat_df.tail(20)

,account,advertId,article_id,date,atbs,canceled,clicks,cpc,cr,ctr,orders,sum,sum_price,views,avg_position,cpm,updated_at
1787,,,,,,,,,,,,,,,,,2026-05-24 0:14
1788,,,,,,,,,,,,,,,,,2026-05-24 0:14
1789,,,,,,,,,,,,,,,,,2026-05-24 0:14
1790,,,,,,,,,,,,,,,,,2026-05-24 0:14
1791,,,,,,,,,,,,,,,,,2026-05-24 0:14
1792,BUENOS ANIMALES,36270150,398400809,2026-05-23,3,0,77,"11,99","1,3","1,74",1,1,"923,5",4282,4422,"208,8421529",2026-05-24 7:35
1793,BUENOS ANIMALES,35713269,546333547,2026-05-23,15,0,55,14,"9,09","2,07",5,5,770,18128,2651,"290,4564315",2026-05-24 7:35
1794,BUENOS ANIMALES,25111586,170598570,2026-05-23,4,0,35,"2,85",0,"6,74",0,0,"99,79",0,519,"192,2736031",2026-05-24 7:35
1795,BUENOS ANIMALES,22802308,170602860,2026-05-23,4,0,26,"3,48","3,85","10,66",1,1,"90,43",2603,244,"370,6147541",2026-05-24 7:35
1796,BUENOS ANIMALES,25431651,159221368,2026-05-23,2,0,21,"6,75",0,"4,45",0,0,"141,78",0,472,"300,3813559",2026-05-24 7:35


In [14]:
advert_stat_df.isna().sum().sort_values(ascending=False)

account         0
advertId        0
article_id      0
date            0
atbs            0
canceled        0
clicks          0
cpc             0
cr              0
ctr             0
orders          0
sum             0
sum_price       0
views           0
avg_position    0
cpm             0
updated_at      0
dtype: int64

In [15]:
def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        hit = normalized.get(candidate.strip().lower())
        if hit is not None:
            return hit
    return None

DATE_COL = find_col(advert_stat_df, ['date', '????'])
ARTICLE_COL = find_col(advert_stat_df, ['article_id', 'nm_id', 'nmId', '??????? WB'])
ADVERT_COL = find_col(advert_stat_df, ['advertId', 'advert_id', 'campaign_id', 'campaignId'])

DATE_COL, ARTICLE_COL, ADVERT_COL

('date', 'article_id', 'advertId')

In [16]:
if DATE_COL:
    unique_dates = pd.to_datetime(advert_stat_df[DATE_COL], errors='coerce').dt.date.dropna().astype(str).unique().tolist()
    print(f'Unique dates count: {len(unique_dates)}')
    unique_dates[:30]
else:
    print('??????? ???? ?? ???????')

Unique dates count: 117


In [17]:
if ARTICLE_COL:
    print(f'Unique article_id count: {advert_stat_df[ARTICLE_COL].astype(str).nunique()}')
    advert_stat_df[ARTICLE_COL].astype(str).drop_duplicates().head(30).tolist()
else:
    print('??????? article_id ?? ???????')

Unique article_id count: 11


## ???????????? ???????? ???????

In [18]:
NUMERIC_CANDIDATES = ['clicks', 'views', 'orders', 'sum', 'sum_price', 'ctr', 'cpc', 'cpm', 'cr', 'avg_position']

def to_numeric_local(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    s = s.str.replace('\xa0', '', regex=False)
    s = s.str.replace(' ', '', regex=False)
    s = s.str.replace('%', '', regex=False)
    s = s.str.replace(',', '.', regex=False)
    return pd.to_numeric(s, errors='coerce')

advert_norm_df = advert_stat_df.copy()
numeric_map: dict[str, str] = {}
for candidate in NUMERIC_CANDIDATES:
    col = find_col(advert_norm_df, [candidate])
    if col:
        numeric_map[candidate] = col
        advert_norm_df[col] = to_numeric_local(advert_norm_df[col])

numeric_map

{'clicks': 'clicks',
 'views': 'views',
 'orders': 'orders',
 'sum': 'sum',
 'sum_price': 'sum_price',
 'ctr': 'ctr',
 'cpc': 'cpc',
 'cpm': 'cpm',
 'cr': 'cr',
 'avg_position': 'avg_position'}

In [19]:
advert_norm_df.dtypes

account             str
advertId            str
article_id          str
date                str
atbs                str
canceled            str
clicks          float64
cpc             float64
cr              float64
ctr             float64
orders          float64
sum             float64
sum_price       float64
views           float64
avg_position    float64
cpm             float64
updated_at          str
dtype: object

In [20]:
advert_norm_df.head(20)

,account,advertId,article_id,date,atbs,canceled,clicks,cpc,cr,ctr,orders,sum,sum_price,views,avg_position,cpm,updated_at
0,Buenos Animales,25431662,159221368,2025-11-28,0,0,1.0,0.00,0.00,169.00,0.0,21.83,0.0,59.0,0.0,370.00,
1,Buenos Animales,22802307,170602860,2025-11-28,12,0,132.0,7.27,3.79,297.00,5.0,760.32,11750.0,4448.0,40.0,170.94,
2,Buenos Animales,22802308,170602860,2025-11-28,21,0,102.0,26.05,6.86,11.43,7.0,912.95,16450.0,892.0,0.0,1023.49,
3,Buenos Animales,22817187,242981834,2025-11-28,2,0,31.0,0.00,3.23,238.00,1.0,401.25,2350.0,1300.0,38.0,308.65,
4,Buenos Animales,22817188,242981834,2025-11-28,2,0,7.0,0.00,14.29,154.00,1.0,269.40,2350.0,455.0,0.0,592.09,
5,Buenos Animales,25111586,170598570,2025-11-28,13,0,178.0,4.16,0.56,625.00,1.0,528.81,2307.0,2847.0,8.0,185.74,
6,Buenos Animales,25111591,170598570,2025-11-28,2,0,15.0,8.10,0.00,765.00,0.0,72.19,0.0,196.0,0.0,368.32,
7,Buenos Animales,25431651,159221368,2025-11-28,9,0,61.0,7.87,1.64,428.00,1.0,219.98,2307.0,1425.0,123.0,154.37,
8,Buenos Animales,22817187,242981834,2025-11-29,9,0,40.0,11.23,10.00,245.00,4.0,580.39,9400.0,1634.0,28.0,355.20,
9,Buenos Animales,22817188,242981834,2025-11-29,8,0,31.0,0.00,9.68,477.00,3.0,317.67,7050.0,650.0,0.0,488.72,


## ??????????

In [21]:
sample_date = None
if DATE_COL:
    sample_date = pd.to_datetime(advert_norm_df[DATE_COL], errors='coerce').dt.date.dropna().astype(str).head(1)
    sample_date = sample_date.iloc[0] if len(sample_date) else None

sample_date

'2025-11-28'

In [22]:
if DATE_COL and sample_date:
    by_date_df = advert_norm_df[pd.to_datetime(advert_norm_df[DATE_COL], errors='coerce').dt.date.astype(str) == sample_date]
    print(f'Rows for date {sample_date}: {len(by_date_df)}')
    by_date_df.head(20)
else:
    print('?????????? ?? ???? ?????????? (??? date ??? ??????)')

Rows for date 2025-11-28: 8


In [23]:
sample_article = None
if ARTICLE_COL:
    sample_article = advert_norm_df[ARTICLE_COL].astype(str).dropna().head(1)
    sample_article = sample_article.iloc[0] if len(sample_article) else None

sample_article

'159221368'

In [24]:
if ARTICLE_COL and sample_article:
    by_article_df = advert_norm_df[advert_norm_df[ARTICLE_COL].astype(str) == str(sample_article)]
    print(f'Rows for article_id {sample_article}: {len(by_article_df)}')
    by_article_df.head(20)
else:
    print('?????????? ?? article_id ?????????? (??? ??????? ??? ??????)')

Rows for article_id 159221368: 220


In [25]:
if ARTICLE_COL and DATE_COL:
    group_cols = [ARTICLE_COL, DATE_COL]
    numeric_cols = [c for c in numeric_map.values() if c in advert_norm_df.columns]
    grouped = advert_norm_df.groupby(group_cols, dropna=False)[numeric_cols].sum(min_count=1).reset_index()
    print(f'Grouped rows: {len(grouped)}')
    grouped.head(30)
else:
    print('??????????? ?? article_id + date ??????????')

Grouped rows: 895


## ??????????????? ????????

In [26]:
if DATE_COL and ADVERT_COL and ARTICLE_COL:
    dup_mask = advert_norm_df.duplicated(subset=[DATE_COL, ADVERT_COL, ARTICLE_COL], keep=False)
    duplicates_df = advert_norm_df[dup_mask].sort_values([DATE_COL, ADVERT_COL, ARTICLE_COL])
    print(f'Duplicates date+advertId+article_id: {len(duplicates_df)}')
    duplicates_df.head(30)
else:
    print('???????? ?????? ??????????: ?? ??????? date/advertId/article_id')

Duplicates date+advertId+article_id: 108


In [27]:
neg_cols = [c for c in numeric_map.values() if c in advert_norm_df.columns]
negative_rows = advert_norm_df[(advert_norm_df[neg_cols] < 0).any(axis=1)] if neg_cols else advert_norm_df.iloc[0:0]
print(f'Negative value rows: {len(negative_rows)}')
negative_rows.head(30)

Negative value rows: 0


,account,advertId,article_id,date,atbs,canceled,clicks,cpc,cr,ctr,orders,sum,sum_price,views,avg_position,cpm,updated_at


In [28]:
ctr_col = numeric_map.get('ctr')
cr_col = numeric_map.get('cr')

anomalous_ctr = advert_norm_df[(advert_norm_df[ctr_col] > 100) | (advert_norm_df[ctr_col] < 0)] if ctr_col else advert_norm_df.iloc[0:0]
anomalous_cr = advert_norm_df[(advert_norm_df[cr_col] > 100) | (advert_norm_df[cr_col] < 0)] if cr_col else advert_norm_df.iloc[0:0]

print(f'Anomalous CTR rows: {len(anomalous_ctr)}')
print(f'Anomalous CR rows: {len(anomalous_cr)}')

Anomalous CTR rows: 1306
Anomalous CR rows: 5


In [29]:
sum_col = numeric_map.get('sum')
clicks_col = numeric_map.get('clicks')
orders_col = numeric_map.get('orders')
sum_price_col = numeric_map.get('sum_price')

spend_zero_clicks_rows = advert_norm_df[(advert_norm_df[sum_col] == 0) & (advert_norm_df[clicks_col] > 0)] if sum_col and clicks_col else advert_norm_df.iloc[0:0]
orders_without_revenue_rows = advert_norm_df[(advert_norm_df[orders_col] > 0) & (advert_norm_df[sum_price_col] <= 0)] if orders_col and sum_price_col else advert_norm_df.iloc[0:0]

print(f'Rows with spend=0 and clicks>0: {len(spend_zero_clicks_rows)}')
print(f'Rows with orders>0 and revenue<=0: {len(orders_without_revenue_rows)}')

Rows with spend=0 and clicks>0: 439
Rows with orders>0 and revenue<=0: 6


In [30]:
spend_zero_clicks_rows.head(20)

,account,advertId,article_id,date,atbs,canceled,clicks,cpc,cr,ctr,orders,sum,sum_price,views,avg_position,cpm,updated_at
981,BUENOS ANIMALES,25111586,170598570,2026-04-04,7,0,88.0,2.24,0.0,732.0,0.0,0.0,196.84,0.0,1203.0,163.62,
983,BUENOS ANIMALES,31687120,546335813,2026-04-04,3,0,54.0,9.69,0.0,409.0,0.0,0.0,523.01,0.0,1320.0,396.22,
984,BUENOS ANIMALES,32663747,703679220,2026-04-04,10,0,48.0,3.73,0.0,369.0,0.0,0.0,178.82,0.0,1302.0,137.34,
985,BUENOS ANIMALES,25431651,159221368,2026-04-04,5,0,38.0,4.55,0.0,531.0,0.0,0.0,172.94,0.0,716.0,241.54,
986,BUENOS ANIMALES,31687103,546335814,2026-04-04,1,0,18.0,14.62,0.0,290.0,0.0,0.0,263.10,0.0,621.0,423.67,
987,BUENOS ANIMALES,22802308,170602860,2026-04-04,1,0,15.0,5.21,0.0,877.0,0.0,0.0,78.09,0.0,171.0,456.67,
988,BUENOS ANIMALES,25431662,159221368,2026-04-04,1,0,13.0,5.81,0.0,833.0,0.0,0.0,75.55,0.0,156.0,484.29,
989,BUENOS ANIMALES,25111591,170598570,2026-04-04,1,0,7.0,7.45,0.0,579.0,0.0,0.0,52.15,0.0,121.0,430.99,
990,BUENOS ANIMALES,32663752,703679220,2026-04-04,1,0,5.0,10.63,0.0,321.0,0.0,0.0,53.13,0.0,156.0,340.58,
992,BUENOS ANIMALES,22817188,242981834,2026-04-04,0,0,1.0,68.35,0.0,79.0,0.0,0.0,68.35,0.0,126.0,542.46,


In [31]:
orders_without_revenue_rows.head(20)

,account,advertId,article_id,date,atbs,canceled,clicks,cpc,cr,ctr,orders,sum,sum_price,views,avg_position,cpm,updated_at
1190,BUENOS ANIMALES,35713269,546333547,2026-04-17,2,0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,3672.0,1.0,0.0,2026-04-18 12:00
1254,BUENOS ANIMALES,35713269,546333547,2026-04-21,0,0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,6000.0,4.0,0.0,2026-04-18 12:00
1322,BUENOS ANIMALES,35713269,546333547,2026-04-26,0,0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,6200.0,3.0,0.0,2026-04-20 7:35
1475,BUENOS ANIMALES,35713269,546333547,2026-05-05,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,3400.0,12.0,0.0,2026-05-01 7:35
1507,BUENOS ANIMALES,35713269,546333547,2026-05-07,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,3300.0,3.0,0.0,2026-05-02 7:35
1685,BUENOS ANIMALES,35463170,546333547,2026-05-20,0,0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,3150.0,0.0,NaN,2026-05-15 7:35
